# HDB Resale Price Regression — Notebook 14: Revised Final Model

Notebooks 12–13 stress-tested Model 10 with five diagnostics and three robustness checks. Two variables did not survive:

- **`price_has_168`** — the "一路发 prosperity all the way" premium (+$32,795 in OLS) collapsed to $17,233 under median regression and lost significance (p = 0.198). The OLS estimate was inflated by a handful of expensive transactions.
- **`cny_month`** — the $59,310 "Chinese New Year month" effect was an aliasing artifact. March 2026 was the only month in the dataset where every transaction fell in a CNY month, making `cny_month` perfectly collinear with the March 2026 month dummy. Once resolved, the true effect is ~$880 and not significant.

This notebook builds **Model 11**: Model 10 minus these two variables. Everything that survived — the lucky-8 premium, the block-4 discount, all distance and structural variables — stays in.

**What survived robustness checks:**
| Variable | OLS | LAD | Cook's D removal | Verdict |
|---|---|---|---|---|
| num_eights_tail | +$1,070 | +$1,371 | +$1,354 | Robust — strengthens |
| block_has_4 | -$10,160 | -$8,721 | -$8,557 | Robust |
| All distance vars | Stable | Stable | Stable | Robust |
| All structural vars | Stable | Stable | Stable | Robust |

In [1]:
%load_ext rpy2.ipython
import warnings
warnings.filterwarnings('ignore')

Error importing in API mode: ImportError("dlopen(/Users/wongpeiting/.pyenv/versions/3.13.9/lib/python3.13/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib\n  Referenced from: <B96A8100-FA7A-3EFC-8726-931D26646DE6> /Users/wongpeiting/.pyenv/versions/3.13.9/lib/python3.13/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file)")


Trying to import in ABI mode.


In [2]:
%%R
library(tidyverse)
library(sandwich)
library(lmtest)
library(car)

df <- read_csv('data/hdb_analysis.csv', show_col_types = FALSE)
df$remaining_lease_sq <- df$remaining_lease_years^2
df$month_factor <- factor(format(df$month, '%Y-%m'))
df$ln_price <- log(df$resale_price)

# Model 11: Model 10 minus price_has_168 and cny_month
model11 <- lm(resale_price ~ town + flat_type + floor_area_sqm + storey_mid +
              remaining_lease_years + remaining_lease_sq +
              flat_model_grouped +
              dist_cbd_km + mrt_dist_m + hawker_dist_m +
              popular_school_dist_m +
              park_dist_m + hospital_dist_m +
              columbarium_dist_m + temple_dist_m +
              coast_dist_m +
              num_eights_tail +
              block_has_4 +
              month_factor,
            data = df)

# Model 10 for comparison
model10 <- lm(resale_price ~ town + flat_type + floor_area_sqm + storey_mid +
              remaining_lease_years + remaining_lease_sq +
              flat_model_grouped +
              dist_cbd_km + mrt_dist_m + hawker_dist_m +
              popular_school_dist_m +
              park_dist_m + hospital_dist_m +
              columbarium_dist_m + temple_dist_m +
              coast_dist_m +
              num_eights_tail +
              price_has_168 +
              block_has_4 +
              cny_month +
              month_factor,
            data = df)

cat(sprintf('Model 10 (original):  R\u00b2 = %.4f, Adj R\u00b2 = %.4f, %d parameters\n',
    summary(model10)$r.squared, summary(model10)$adj.r.squared, length(coef(model10))))
cat(sprintf('Model 11 (revised):   R\u00b2 = %.4f, Adj R\u00b2 = %.4f, %d parameters\n',
    summary(model11)$r.squared, summary(model11)$adj.r.squared, length(coef(model11))))
cat(sprintf('R\u00b2 change:            %+.4f\n',
    summary(model11)$r.squared - summary(model10)$r.squared))
cat(sprintf('Observations:         %s\n', format(nrow(df), big.mark = ',')))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.1     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Model 10 (original):  R² = 0.9023, Adj R² = 0.9021, 87 parameters


Model 11 (revised):   R² = 0.9022, Adj R² = 0.9021, 85 parameters


R² change:            -0.0001


Observations:         50,718


Loading required package: zoo

Attaching package: ‘zoo’

The following objects are masked from ‘package:base’:

    as.Date, as.Date.numeric

Loading required package: carData

Attaching package: ‘car’

The following object is masked from ‘package:dplyr’:

    recode

The following object is masked from ‘package:purrr’:

    some



## Model 11: full coefficient table

All coefficients with HC1 robust standard errors. Town dummies, flat model dummies, and month fixed effects listed separately.

In [3]:
%%R
robust11 <- coeftest(model11, vcov = vcovHC(model11, type = 'HC1'))

# Key variables (non-dummy)
key_vars <- c('(Intercept)', 'floor_area_sqm', 'storey_mid',
              'remaining_lease_years', 'remaining_lease_sq',
              'dist_cbd_km', 'mrt_dist_m', 'hawker_dist_m',
              'popular_school_dist_m',
              'park_dist_m', 'hospital_dist_m',
              'columbarium_dist_m', 'temple_dist_m',
              'coast_dist_m',
              'num_eights_tail',
              'block_has_4')

cat(sprintf('%-30s %12s %10s %8s\n', 'Variable', 'Coefficient', 'Robust SE', 'p-value'))
cat(paste(rep('=', 63), collapse = ''), '\n')

for (v in key_vars) {
    if (v %in% rownames(robust11)) {
        coef_val <- robust11[v, 'Estimate']
        se <- robust11[v, 'Std. Error']
        p <- robust11[v, 'Pr(>|t|)']
        sig <- ifelse(p < 0.001, '***', ifelse(p < 0.01, '**', ifelse(p < 0.05, '*', '')))
        cat(sprintf('%-30s $%+10.0f %10.0f %7.4f %s\n', v, coef_val, se, p, sig))
    }
}

# Flat type dummies
cat(sprintf('\n--- Flat type (ref: 1 ROOM) ---\n'))
ft_rows <- grep('^flat_type', rownames(robust11))
for (i in ft_rows) {
    v <- rownames(robust11)[i]
    label <- gsub('flat_type', '', v)
    coef_val <- robust11[v, 'Estimate']
    p <- robust11[v, 'Pr(>|t|)']
    sig <- ifelse(p < 0.001, '***', ifelse(p < 0.01, '**', ifelse(p < 0.05, '*', '')))
    cat(sprintf('  %-28s $%+10.0f %7.4f %s\n', label, coef_val, p, sig))
}

# Flat model dummies
cat(sprintf('\n--- Flat model (ref: 2-room) ---\n'))
fm_rows <- grep('^flat_model_grouped', rownames(robust11))
for (i in fm_rows) {
    v <- rownames(robust11)[i]
    label <- gsub('flat_model_grouped', '', v)
    coef_val <- robust11[v, 'Estimate']
    p <- robust11[v, 'Pr(>|t|)']
    sig <- ifelse(p < 0.001, '***', ifelse(p < 0.01, '**', ifelse(p < 0.05, '*', '')))
    cat(sprintf('  %-28s $%+10.0f %7.4f %s\n', label, coef_val, p, sig))
}

# Town dummies
cat(sprintf('\n--- Town (ref: ANG MO KIO) ---\n'))
town_rows <- grep('^town', rownames(robust11))
for (i in town_rows) {
    v <- rownames(robust11)[i]
    label <- gsub('town', '', v)
    coef_val <- robust11[v, 'Estimate']
    p <- robust11[v, 'Pr(>|t|)']
    sig <- ifelse(p < 0.001, '***', ifelse(p < 0.01, '**', ifelse(p < 0.05, '*', '')))
    cat(sprintf('  %-28s $%+10.0f %7.4f %s\n', label, coef_val, p, sig))
}

cat(sprintf('\n+ %d month fixed effects (omitted)\n',
    length(grep('month_factor', rownames(robust11)))))

# F-statistic
s <- summary(model11)
cat(sprintf('\nF-statistic: %.1f on %d and %d DF, p-value: %s\n',
    s$fstatistic[1], s$fstatistic[2], s$fstatistic[3],
    format.pval(pf(s$fstatistic[1], s$fstatistic[2], s$fstatistic[3], lower.tail=FALSE))))

Variable                        Coefficient  Robust SE  p-value


(Intercept)                    $   -240525      17401  0.0000 ***


floor_area_sqm                 $     +5425        160  0.0000 ***


storey_mid                     $     +5425         69  0.0000 ***


remaining_lease_years          $    +11358        367  0.0000 ***


...

  TOA PAYOH                    $     -5888  0.1088 


  WOODLANDS                    $    -84482  0.0000 ***


  YISHUN                       $    -13945  0.0017 **



+ 23 month fixed effects (omitted)



F-statistic: 5562.3 on 84 and 50633 DF, p-value: < 2.22e-16


## Log model equivalent

Same Model 11 variables with ln(resale_price) as the dependent variable, plus the floor_area × Terrace interaction from Notebook 11.

In [4]:
%%R
df$is_terrace <- ifelse(df$flat_model_grouped == 'Terrace', 1, 0)

model11_log <- lm(ln_price ~ town + flat_type + floor_area_sqm * is_terrace + storey_mid +
              remaining_lease_years + remaining_lease_sq +
              flat_model_grouped +
              dist_cbd_km + mrt_dist_m + hawker_dist_m +
              popular_school_dist_m +
              park_dist_m + hospital_dist_m +
              columbarium_dist_m + temple_dist_m +
              coast_dist_m +
              num_eights_tail +
              block_has_4 +
              month_factor,
            data = df)

cat(sprintf('Model 11 (raw):   R\u00b2 = %.4f\n', summary(model11)$r.squared))
cat(sprintf('Model 11 (log):   R\u00b2 = %.4f\n', summary(model11_log)$r.squared))

# Compare predictions on dollar scale
df$pred_raw <- predict(model11, df)
df$pred_log <- exp(predict(model11_log, df))

cat(sprintf('\nPrediction comparison (dollar scale):\n'))
cat(sprintf('  %-25s %12s %12s\n', '', 'Raw model', 'Log model'))
cat(paste(rep('-', 50), collapse = ''), '\n')
cat(sprintf('  %-25s $%10s $%10s\n', 'Mean absolute error',
    format(round(mean(abs(df$resale_price - df$pred_raw))), big.mark = ','),
    format(round(mean(abs(df$resale_price - df$pred_log))), big.mark = ',')))
cat(sprintf('  %-25s $%10s $%10s\n', 'Median absolute error',
    format(round(median(abs(df$resale_price - df$pred_raw))), big.mark = ','),
    format(round(median(abs(df$resale_price - df$pred_log))), big.mark = ',')))
cat(sprintf('  %-25s %12d %12d\n', 'Negative predictions',
    sum(df$pred_raw < 0), sum(df$pred_log < 0)))

# Key coefficients comparison
cat('\nKey coefficients (log model, approximate % interpretation):\n')
robust_log <- coeftest(model11_log, vcov = vcovHC(model11_log, type = 'HC1'))
log_vars <- c('floor_area_sqm', 'storey_mid', 'remaining_lease_years',
              'dist_cbd_km', 'mrt_dist_m', 'columbarium_dist_m', 'temple_dist_m',
              'num_eights_tail', 'block_has_4')

cat(sprintf('%-25s %12s %12s\n', 'Variable', 'Raw ($)', 'Log (%)'))
cat(paste(rep('-', 50), collapse = ''), '\n')
for (v in log_vars) {
    if (v %in% rownames(robust11) & v %in% rownames(robust_log)) {
        c_raw <- robust11[v, 'Estimate']
        c_log <- robust_log[v, 'Estimate'] * 100
        cat(sprintf('%-25s $%+10.0f %+10.3f%%\n', v, c_raw, c_log))
    }
}

Model 11 (raw):   R² = 0.9022


Model 11 (log):   R² = 0.9379



Prediction comparison (dollar scale):


                               Raw model    Log model


--------------------------------------------------

...

mrt_dist_m                $       -80     -0.012%


columbarium_dist_m        $        +8     +0.001%


temple_dist_m             $       -25     -0.003%


num_eights_tail           $     +1189     +0.246%


block_has_4               $    -10159     -1.361%


## Lucky-8 premium by price level

The lucky-8 premium is not constant — it reverses for the cheapest flats and grows with price. Price quartiles are based on predicted price *without* superstition variables, to avoid endogeneity.

In [5]:
%%R
# Predicted price without superstition variables
model_no_super <- lm(resale_price ~ town + flat_type + floor_area_sqm + storey_mid +
              remaining_lease_years + remaining_lease_sq +
              flat_model_grouped +
              dist_cbd_km + mrt_dist_m + hawker_dist_m +
              popular_school_dist_m +
              park_dist_m + hospital_dist_m +
              columbarium_dist_m + temple_dist_m +
              coast_dist_m +
              month_factor,
            data = df)

df$pred_no_super <- predict(model_no_super, df)
df$price_quartile <- cut(df$pred_no_super,
    breaks = quantile(df$pred_no_super, probs = c(0, 0.25, 0.5, 0.75, 1)),
    labels = c('Q1', 'Q2', 'Q3', 'Q4'),
    include.lowest = TRUE)

# Model 11 with interaction
model11_interact <- lm(resale_price ~ town + flat_type + floor_area_sqm + storey_mid +
              remaining_lease_years + remaining_lease_sq +
              flat_model_grouped +
              dist_cbd_km + mrt_dist_m + hawker_dist_m +
              popular_school_dist_m +
              park_dist_m + hospital_dist_m +
              columbarium_dist_m + temple_dist_m +
              coast_dist_m +
              num_eights_tail * price_quartile +
              block_has_4 +
              month_factor,
            data = df)

robust_int <- coeftest(model11_interact, vcov = vcovHC(model11_interact, type = 'HC1'))

base_8 <- robust_int['num_eights_tail', 'Estimate']
cat(sprintf('Lucky-8 premium by price quartile:\n\n'))
cat(sprintf('%-30s %10s %10s %12s\n', 'Quartile', 'Price range', '8-premium', 'p-value'))
cat(paste(rep('-', 65), collapse = ''), '\n')

quartile_labels <- c('Q1 (cheapest 25%)', 'Q2', 'Q3', 'Q4 (most expensive 25%)')
for (j in 1:4) {
    q <- c('Q1', 'Q2', 'Q3', 'Q4')[j]
    subset <- df[df$price_quartile == q, ]
    price_range <- sprintf('$%s-$%s',
        format(round(min(subset$pred_no_super) / 1000), big.mark = ','),
        format(round(max(subset$pred_no_super) / 1000), big.mark = ','))

    if (q == 'Q1') {
        total <- base_8
        p_val <- robust_int['num_eights_tail', 'Pr(>|t|)']
    } else {
        int_name <- sprintf('num_eights_tail:price_quartile%s', q)
        if (int_name %in% rownames(robust_int)) {
            total <- base_8 + robust_int[int_name, 'Estimate']
            p_val <- robust_int[int_name, 'Pr(>|t|)']
        }
    }
    sig <- ifelse(p_val < 0.05, '*', '')
    cat(sprintf('%-30s %10s $%+8.0f %10.4f %s\n',
        quartile_labels[j], price_range, total, p_val, sig))
}

Lucky-8 premium by price quartile:



Quartile                       Price range  8-premium      p-value


-----------------------------------------------------------------

Q1 (cheapest 25%)                $-3-$509 $   -1795     0.0006 *


Q2                              $509-$641 $   +1482     0.0000 *


Q3                              $641-$761 $   +1955     0.0000 *


Q4 (most expensive 25%)        $761-$2,302 $   +2166     0.0000 *


## Revised outlier lists

Alamak flats (sold way above prediction) and bargain flats (sold way below) from Model 11.

In [6]:
%%R
df$pred11 <- predict(model11, df)
df$resid11 <- df$resale_price - df$pred11
df$resid11_pct <- round(df$resid11 / df$pred11 * 100, 1)

cat('=== TOP 20 ALAMAK FLATS (sold way above Model 11 prediction) ===\n\n')
cat(sprintf('%-5s %-30s %-10s %-8s %10s %10s %8s\n',
    'Rank', 'Address', 'Type', 'Storey', 'Actual', 'Predicted', 'Error'))
cat(paste(rep('-', 85), collapse = ''), '\n')

alamak <- df[order(-df$resid11), ]
for (i in 1:20) {
    r <- alamak[i, ]
    label <- sprintf('Blk %s %s', r$block, substr(r$street_name, 1, 20))
    cat(sprintf('%-5d %-30s %-10s %-8s $%9s $%9s %+7.1f%%\n',
        i, label, r$flat_type, r$storey_range,
        format(round(r$resale_price), big.mark = ','),
        format(round(r$pred11), big.mark = ','),
        r$resid11_pct))
}

cat('\n\n=== TOP 20 BARGAIN FLATS (sold way below Model 11 prediction) ===\n\n')
cat(sprintf('%-5s %-30s %-10s %-8s %10s %10s %8s\n',
    'Rank', 'Address', 'Type', 'Storey', 'Actual', 'Predicted', 'Error'))
cat(paste(rep('-', 85), collapse = ''), '\n')

bargain <- df[order(df$resid11), ]
for (i in 1:20) {
    r <- bargain[i, ]
    label <- sprintf('Blk %s %s', r$block, substr(r$street_name, 1, 20))
    cat(sprintf('%-5d %-30s %-10s %-8s $%9s $%9s %+7.1f%%\n',
        i, label, r$flat_type, r$storey_range,
        format(round(r$resale_price), big.mark = ','),
        format(round(r$pred11), big.mark = ','),
        r$resid11_pct))
}

=== TOP 20 ALAMAK FLATS (sold way above Model 11 prediction) ===



Rank  Address                        Type       Storey       Actual  Predicted    Error


-------------------------------------------------------------------------------------

1     Blk 9A BOON TIONG RD           5 ROOM     25 TO 27 $1,648,888 $1,195,945   +37.9%


2     Blk 241 BISHAN ST 22           EXECUTIVE  07 TO 09 $1,448,000 $  999,249   +44.9%


...

16    Blk 997C BUANGKOK CRES         5 ROOM     16 TO 18 $  768,000 $1,041,239   -26.2%


17    Blk 997C BUANGKOK CRES         5 ROOM     13 TO 15 $  790,000 $1,061,341   -25.6%


18    Blk 997B BUANGKOK CRES         5 ROOM     07 TO 09 $  735,000 $1,003,325   -26.7%


19    Blk 997C BUANGKOK CRES         5 ROOM     10 TO 12 $  775,000 $1,042,721   -25.7%


20    Blk 997A BUANGKOK CRES         5 ROOM     07 TO 09 $  760,020 $1,024,128   -25.8%


## Summary: Model 11

### What changed from Model 10

| | Model 10 | Model 11 |
|---|---|---|
| R² | 0.9023 | 0.9022 |
| Adj R² | 0.9021 | 0.9021 |
| Parameters | 87 | 85 |
| Variables dropped | — | `price_has_168`, `cny_month` |
| R² change | — | −0.0001 |

Dropping two variables that failed robustness checks costs essentially nothing: R² falls by 0.0001. Adjusted R² is identical. The model is tighter and more defensible.

### What drives HDB resale prices (revised)

**Location**
- Distance to CBD: $−16,110 per km (holding everything else constant, a flat 1 km further from Raffles Place is worth ~$16K less)
- MRT proximity: $−80 per metre; a flat 500m from the nearest station is worth ~$40K less than one outside the exit
- Hawker centre: $−20 per metre; $10K penalty for every 500m
- Popular primary school: $−10 per metre
- Temple noise: $−25 per metre away (closer to temple = lower price)
- Columbarium proximity: $+8 per metre (slight premium for distance, i.e. closer = cheaper)

**Lease and physical**
- Floor area: $+5,425 per sqm
- Storey: $+5,425 per mid-storey level (~$5K per floor)
- Remaining lease: $+11,358 per year, with a quadratic decay term ($−31 per year²) that captures the accelerating discount as leases approach expiry
- Town premium vs Ang Mo Kio (reference): Bukit Timah +$184,589; Bishan +$83,935; Queenstown +$15,400; Sengkang −$198,411; Punggol −$176,622; Bukit Panjang −$129,371

**Superstition (survived robustness checks)**
- Lucky-8 digit premium: $+1,189 per trailing 8 in the resale price (p < 0.001). Robust to median regression ($+1,371 in LAD) and Cook's D outlier removal ($+1,354).
- Block-number-4 discount: $−10,159 for blocks containing the digit 4 (p < 0.001). Robust to LAD ($−8,721) and outlier removal ($−8,557).

**Log model (for % interpretation)**
- Each trailing 8 in the price is worth +0.246% of flat value
- Each floor up is +0.713%
- Each additional sqm is +0.883%
- Each year of remaining lease is +2.105%
- Each km from CBD is −2.274%
- Block-4 discount is −1.361% of flat value

### What was dropped and why

- **`price_has_168`:** The $32,795 OLS premium collapsed to $17,233 under median (LAD) regression and lost significance entirely (p = 0.198), indicating the OLS estimate was inflated by a small number of very expensive transactions with that price ending. Not a robust finding.
- **`cny_month`:** The apparent $59,310 "Chinese New Year month" premium in earlier notebooks was an aliasing artifact: March 2026 was the only month in the dataset where every transaction fell in a CNY month, making `cny_month` perfectly collinear with the March 2026 month dummy. Once that collinearity is resolved, the true CNY effect is ~$880 — not distinguishable from zero.

### The lucky-8 effect, refined

The average Model 11 coefficient of $+1,189 per trailing 8 masks sharp variation by price level. In the cheapest 25% of the market (predicted prices below ~$509K), trailing 8s are associated with a statistically significant *discount* of −$1,795 (p < 0.001) — sellers price-signalling with 8s in the bottom quartile actually leave money on the table. The premium only emerges in the middle market: +$1,482 for Q2 flats ($509K–$641K), +$1,955 for Q3 ($641K–$761K), and +$2,166 for the top quartile (above $761K). This means the lucky-8 premium is a behaviour concentrated among buyers of expensive flats, not a universal cultural reflex — trailing 8s are valued by buyers with pricing power who can afford to meet a psychologically curated price, while budget-flat buyers comparison-shop and discount sellers who appear to be gaming their price endings.